# TrackNet-Pickleball: Complete Pipeline on Google Colab

This notebook runs the **entire** TrackNet-Pickleball pipeline end-to-end on Google Colab.

---

## Two Modes of Operation

### Mode 1: INFERENCE ONLY (Just detect the ball — no training needed)
**Use this if you only have a video (e.g. downloaded from YouTube) and NO label CSV.**
1. Run Sections 0-2 (setup, clone, download weights)
2. Upload your video (Section 3)
3. Extract frames (Step 1)
4. **Set `SKIP_TO_INFERENCE = True` in the cell below Step 2-3**
5. Run Step 6 (predictions) → Step 8 (trajectory video)
6. Download results (Section 9)

### Mode 2: FULL TRAINING PIPELINE
**Use this if you have BOTH a video AND a manually-created label CSV with ball positions.**
A label CSV has columns: `Frame, Visibility, X, Y` — where X,Y are pixel coordinates of the ball.
You create this by manually clicking on the ball in every single frame using a labelling tool.
1. Run ALL sections in order

---

**Before running:** Go to `Runtime > Change runtime type` and select **GPU** (T4 is fine).

## 0. Environment Setup

In [ ]:
# Verify GPU is available
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU")

In [ ]:
# Install required packages (most are pre-installed on Colab)
!pip install -q opencv-python-headless pillow scikit-learn gdown

In [ ]:
import numpy as np
import pandas as pd
import os
import sys
import cv2
import math
import csv
import gc
import time
import queue
import shutil
import matplotlib.pyplot as plt
import tensorflow as tf
from glob import glob
from PIL import Image, ImageDraw
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import array_to_img, img_to_array, load_img
from keras.models import Model, load_model
from keras.layers import (
    Input, Conv2D, Activation, BatchNormalization,
    MaxPooling2D, UpSampling2D, concatenate
)
import keras.backend as K
from keras import optimizers

%matplotlib inline
print("All imports successful!")

## 1. Clone Repo & Set Up Working Directory

In [ ]:
# Mount Google Drive (for saving weights, videos, data persistently)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repository
REPO_DIR = '/content/TrackNet-Pickleball'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/AndrewDettor/TrackNet-Pickleball.git {REPO_DIR}
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Create working directories
WORK_DIR = '/content/tracknet_workspace'
FRAMES_DIR = os.path.join(WORK_DIR, 'frames')
DATA_DIR = os.path.join(WORK_DIR, 'npy_data')
WEIGHTS_DIR = os.path.join(WORK_DIR, 'weights')
PREDICTIONS_DIR = os.path.join(WORK_DIR, 'predictions')
TRAJECTORY_DIR = os.path.join(WORK_DIR, 'trajectories')
VIDEOS_DIR = os.path.join(WORK_DIR, 'videos')
LABELS_DIR = os.path.join(WORK_DIR, 'labels')

for d in [WORK_DIR, FRAMES_DIR, DATA_DIR, WEIGHTS_DIR, PREDICTIONS_DIR, TRAJECTORY_DIR, VIDEOS_DIR, LABELS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Workspace directories created at:", WORK_DIR)

## 2. Download Pre-trained Weights

In [ ]:
import gdown

# --- Old Weights (TrackNetV2 trained on badminton) ---
OLD_WEIGHTS_PATH = os.path.join(WEIGHTS_DIR, 'old_weights')
if not os.path.exists(OLD_WEIGHTS_PATH):
    print("Downloading old TrackNetV2 weights (badminton)...")
    # This is a single file; gdown downloads from Google Drive
    gdown.download(
        'https://drive.google.com/uc?id=16ZnOljaxW6zM4bP7TTo1t81gaty7Egts',
        OLD_WEIGHTS_PATH + '.zip',
        quiet=False
    )
    !unzip -q {OLD_WEIGHTS_PATH}.zip -d {OLD_WEIGHTS_PATH}
    print("Old weights downloaded.")
else:
    print("Old weights already exist.")

# --- New Weights (fine-tuned on pickleball) ---
NEW_WEIGHTS_PATH = os.path.join(WEIGHTS_DIR, 'new_weights')
if not os.path.exists(NEW_WEIGHTS_PATH):
    print("Downloading new pickleball weights...")
    gdown.download_folder(
        'https://drive.google.com/drive/folders/1EGsddY1fgEJ5ITrfF32aPCn6nml2Anzr',
        output=NEW_WEIGHTS_PATH,
        quiet=False
    )
    print("New weights downloaded.")
else:
    print("New weights already exist.")

print("\nWeights directory contents:")
!ls -la {WEIGHTS_DIR}
print()
!ls -la {WEIGHTS_DIR}/*

## 3. Upload Your Pickleball Video

You have two options:
- **Option A:** Upload from your computer
- **Option B:** Copy from Google Drive

In [ ]:
# ========================
# Option A: Upload from computer
# ========================
from google.colab import files
print("Upload your pickleball video (.mp4):")
uploaded = files.upload()
for filename in uploaded.keys():
    src = filename
    dst = os.path.join(VIDEOS_DIR, filename)
    shutil.move(src, dst)
    print(f"Moved {filename} to {dst}")

In [ ]:
# ========================
# Option B: Copy from Google Drive (uncomment and edit path)
# ========================
# DRIVE_VIDEO_PATH = '/content/drive/MyDrive/your_video.mp4'
# shutil.copy(DRIVE_VIDEO_PATH, os.path.join(VIDEOS_DIR, os.path.basename(DRIVE_VIDEO_PATH)))
# print(f"Copied video to {VIDEOS_DIR}")

In [ ]:
# ============================================================
# IMPORTANT: Set your actual video filename here!
# After uploading in the cell above, replace 'your_video.mp4'
# with the exact name of YOUR file (e.g. 'pickleball_match.mp4')
# ============================================================
VIDEO_FILENAME = 'your_video.mp4'  # <--- CHANGE THIS to your actual filename
VIDEO_PATH = os.path.join(VIDEOS_DIR, VIDEO_FILENAME)

assert os.path.isfile(VIDEO_PATH), f"Video not found at {VIDEO_PATH}. Check the filename."

cap = cv2.VideoCapture(VIDEO_PATH)
fps = int(cap.get(cv2.CAP_PROP_FPS))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"Video: {VIDEO_FILENAME}")
print(f"Resolution: {width}x{height}")
print(f"FPS: {fps}")
print(f"Total frames: {n_frames}")
print(f"Duration: {n_frames/fps:.1f} seconds")

---
## Step 1: Video to Frames

In [ ]:
VIDEO_FRAMES_DIR = os.path.join(FRAMES_DIR, VIDEO_FILENAME[:-4])

if os.path.exists(VIDEO_FRAMES_DIR):
    shutil.rmtree(VIDEO_FRAMES_DIR)
os.makedirs(VIDEO_FRAMES_DIR)

cap = cv2.VideoCapture(VIDEO_PATH)
success, count = True, 0
success, image = cap.read()
while success:
    cv2.imwrite(os.path.join(VIDEO_FRAMES_DIR, f'{count}.png'), image)
    count += 1
    success, image = cap.read()
    if count % 500 == 0:
        print(f'  Extracted {count} frames...')

cap.release()
print(f'Done! Extracted {count} frames to {VIDEO_FRAMES_DIR}')

---
## Step 2 & 3: Labelling

The original labelling tool uses OpenCV GUI (`cv2.imshow`), which doesn't work in Colab.

**You have 3 options:**

- **Option A (Recommended):** Upload a pre-made label CSV
- **Option B:** Use the inline Colab labelling widget below (simplified)
- **Option C:** Skip labelling and just run inference with pre-trained weights (jump to Step 6)

In [ ]:
# ========================
# Option A: Upload a pre-made label CSV
# ========================
# The CSV should have columns: Frame, Visibility, X, Y
# where X, Y are pixel coordinates and Visibility is 0 or 1

from google.colab import files
print("Upload your label CSV:")
uploaded_labels = files.upload()
for filename in uploaded_labels.keys():
    LABEL_CSV_PATH = os.path.join(LABELS_DIR, 'labels_fixed.csv')
    shutil.move(filename, LABEL_CSV_PATH)
    print(f"Label CSV saved to {LABEL_CSV_PATH}")

In [ ]:
# ========================
# Option B: Simple Colab Labelling Tool
# Click on the ball in each frame, or press 'Skip' if no ball visible
# ========================
# Uncomment and run this cell to use the interactive labelling tool

# from IPython.display import display, clear_output
# from google.colab import output
# import ipywidgets as widgets
# 
# label_data = []
# frame_files = sorted(glob(os.path.join(VIDEO_FRAMES_DIR, '*.png')), key=lambda x: int(os.path.basename(x)[:-4]))
# total_frames = len(frame_files)
# current_frame = [0]  # use list for mutability in closure
# 
# def show_frame(idx):
#     clear_output(wait=True)
#     img = Image.open(frame_files[idx])
#     img_resized = img.resize((640, 360))
#     print(f"Frame {idx}/{total_frames-1} - Click on ball or press Skip")
#     display(img_resized)
# 
# print("For Colab labelling, it's easier to label on your local machine")
# print("and upload the CSV. See Option A above.")

In [ ]:
# ================================================================
# IMPORTANT: Choose your mode here!
# ================================================================
#
# Set to True  → INFERENCE ONLY (you just have a video, no labels)
#                 Skips training, uses pre-trained weights to detect ball.
#                 This is what you want for a YouTube video.
#
# Set to False → FULL TRAINING (you have BOTH video AND a label CSV
#                 with manually-marked ball positions for every frame)
# ================================================================
SKIP_TO_INFERENCE = True  # <--- True for YouTube videos / no labels

if SKIP_TO_INFERENCE:
    print("MODE: Inference only — will use pre-trained weights to detect ball.")
    print("Skip ahead to Step 6 cells after extracting frames.")
else:
    print("MODE: Full training — make sure you upload a label CSV in the cell above.")

In [ ]:
# Fix Labels (Step 3)
# If your label CSV came from the labelling tool (columns: Frame, Ball, x, y with normalized coords),
# this cell converts it to the expected format (Frame, Visibility, X, Y with pixel coords).
# If your CSV already has Frame, Visibility, X, Y — skip this cell.

if not SKIP_TO_INFERENCE:
    RAW_LABEL_PATH = LABEL_CSV_PATH  # from the upload step
    FIXED_LABEL_PATH = os.path.join(LABELS_DIR, 'labels_fixed.csv')
    
    # Check if labels need fixing by looking at columns
    df_check = pd.read_csv(RAW_LABEL_PATH, nrows=2)
    if 'Visibility' in df_check.columns and 'X' in df_check.columns:
        print("Labels already in correct format (Frame, Visibility, X, Y). No fixing needed.")
        FIXED_LABEL_PATH = RAW_LABEL_PATH
    else:
        print("Fixing label format...")
        with open(RAW_LABEL_PATH) as csvfile:
            readCSV = csv.reader(csvfile, delimiter=',')
            frames, x, y = [], [], []
            rows = list(readCSV)
            for i in range(1, len(rows)):
                frames.append(int(rows[i][0]))
                x.append(int(rows[i][1]))
                y.append(int(rows[i][2]))

        visibility = [1] * len(frames)
        df_label = pd.DataFrame({'Frame': frames, 'Visibility': visibility, 'X': x, 'Y': y})

        for i in range(frames[-1] + 1):
            if i not in list(df_label['Frame']):
                df_label = pd.concat([df_label, pd.DataFrame({'Frame': [i], 'Visibility': [0], 'X': [0], 'Y': [0]})],
                                     ignore_index=True)

        df_label = df_label.sort_values(by=['Frame']).reset_index(drop=True)
        df_label.to_csv(FIXED_LABEL_PATH, index=False)
        print(f"Fixed labels saved to {FIXED_LABEL_PATH}")
    
    print(f"\nLabel preview:")
    print(pd.read_csv(FIXED_LABEL_PATH).head(10))
    print(f"Total labeled frames: {len(pd.read_csv(FIXED_LABEL_PATH))}")

---
## Step 4: Generate .npy Training Data

Converts frames + label CSV into numpy arrays for training.
Each sample is 3 consecutive frames stacked into `(9, 288, 512)` with corresponding heatmaps `(3, 288, 512)`.

In [ ]:
HEIGHT = 288
WIDTH = 512
SIGMA = 2.5
MAG = 1
NPY_BATCH_SIZE = 250  # number of samples per .npy file

def genHeatMap(w, h, cx, cy, r, mag):
    """Generate a binary circular heatmap centered at (cx, cy)."""
    if cx < 0 or cy < 0:
        return np.zeros((h, w))
    x, y = np.meshgrid(np.linspace(1, w, w), np.linspace(1, h, h))
    heatmap = ((y - (cy + 1))**2) + ((x - (cx + 1))**2)
    heatmap[heatmap <= r**2] = 1
    heatmap[heatmap > r**2] = 0
    return heatmap * mag

In [ ]:
if not SKIP_TO_INFERENCE:
    # Clean data directory
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)
    os.makedirs(DATA_DIR)

    # Read label data
    data = pd.read_csv(FIXED_LABEL_PATH)
    no = data['Frame'].values
    v = data['Visibility'].values
    x_coords = data['X'].values
    y_coords = data['Y'].values
    num = no.shape[0]

    # Get original image size to compute ratio
    sample_img = img_to_array(load_img(os.path.join(VIDEO_FRAMES_DIR, '1.png')))
    ratio = sample_img.shape[0] / HEIGHT

    i = 0
    ptr = 0
    count = 1

    print('Generating .npy training data...')
    print('=' * 60)
    while ptr < num - 2:
        x_data_tmp = []
        y_data_tmp = []
        while (i < ptr + NPY_BATCH_SIZE) and (i < num - 2):
            if no[i] + 2 != no[i + 2]:
                i += 1
                continue

            unit = []
            for j in range(3):
                target = str(no[i + j]) + '.png'
                png_path = os.path.join(VIDEO_FRAMES_DIR, target)
                a = load_img(png_path)
                a = np.moveaxis(img_to_array(a.resize(size=(WIDTH, HEIGHT))), -1, 0)
                unit.append(a[0])
                unit.append(a[1])
                unit.append(a[2])
                del a
            x_data_tmp.append(unit)
            del unit

            unit = []
            for j in range(3):
                if v[i + j] == 0:
                    unit.append(genHeatMap(WIDTH, HEIGHT, -1, -1, SIGMA, MAG))
                else:
                    unit.append(genHeatMap(WIDTH, HEIGHT, int(x_coords[i + j] / ratio), int(y_coords[i + j] / ratio), SIGMA, MAG))
            y_data_tmp.append(unit)
            del unit

            i += 1

        x_data = np.asarray(x_data_tmp, dtype='float32') / 255.0
        del x_data_tmp
        y_data = np.asarray(y_data_tmp)
        del y_data_tmp

        np.save(os.path.join(DATA_DIR, f'x_data_{count}.npy'), x_data)
        print(f'  x_data_{count}.npy (shape: {x_data.shape})')
        np.save(os.path.join(DATA_DIR, f'y_data_{count}.npy'), y_data)
        print(f'  y_data_{count}.npy (shape: {y_data.shape})')
        count += 1
        del x_data, y_data
        ptr = i

    print('=' * 60)
    print(f'Done! Generated {count - 1} batch files in {DATA_DIR}')
else:
    print("Skipping data generation (SKIP_TO_INFERENCE=True)")

---
## Step 5: Transfer Learning (Training)

Loads pre-trained TrackNetV2 weights (badminton), freezes early layers, and fine-tunes the last `k` conv layers on pickleball data.

**Requires GPU.**

In [ ]:
# ---- Model Definition ----

def custom_loss(y_true, y_pred):
    """Focal-style loss for heatmap prediction."""
    loss = (-1) * (
        K.square(1 - y_pred) * y_true * K.log(K.clip(y_pred, K.epsilon(), 1)) +
        K.square(y_pred) * (1 - y_true) * K.log(K.clip(1 - y_pred, K.epsilon(), 1))
    )
    return K.mean(loss)


def TrackNet3(input_height, input_width):
    """TrackNet3: U-Net style encoder-decoder for ball tracking.
    Input:  (9, H, W) — 3 consecutive RGB frames stacked channel-wise
    Output: (3, H, W) — 3 heatmaps (one per frame)
    """
    imgs_input = Input(shape=(9, input_height, input_width))

    # Encoder
    x = Conv2D(64, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(imgs_input)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)
    x = Conv2D(64, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x1 = BatchNormalization()(x)

    x = MaxPooling2D((2,2), strides=(2,2), data_format='channels_first')(x1)
    x = Conv2D(128, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)
    x = Conv2D(128, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x2 = BatchNormalization()(x)

    x = MaxPooling2D((2,2), strides=(2,2), data_format='channels_first')(x2)
    x = Conv2D(256, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)
    x = Conv2D(256, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)
    x = Conv2D(256, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x3 = BatchNormalization()(x)

    x = MaxPooling2D((2,2), strides=(2,2), data_format='channels_first')(x3)
    x = Conv2D(512, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)
    x = Conv2D(512, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)
    x = Conv2D(512, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)

    # Decoder with skip connections
    x = concatenate([UpSampling2D((2,2), data_format='channels_first')(x), x3], axis=1)
    x = Conv2D(256, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)
    x = Conv2D(256, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)
    x = Conv2D(256, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)

    x = concatenate([UpSampling2D((2,2), data_format='channels_first')(x), x2], axis=1)
    x = Conv2D(128, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)
    x = Conv2D(128, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)

    x = concatenate([UpSampling2D((2,2), data_format='channels_first')(x), x1], axis=1)
    x = Conv2D(64, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)
    x = Conv2D(64, (3,3), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('relu')(x)
    x = BatchNormalization()(x)

    x = Conv2D(3, (1,1), kernel_initializer='random_uniform', padding='same', data_format='channels_first')(x)
    x = Activation('sigmoid')(x)

    model = Model(imgs_input, x)
    model.outputWidth = model.output_shape[3]
    model.outputHeight = model.output_shape[2]
    return model

print("Model and loss function defined.")

In [ ]:
# ---- Evaluation helpers ----

def outcome(y_pred, y_true, tol):
    """Compute TP, TN, FP1, FP2, FN from heatmap predictions vs ground truth."""
    n = y_pred.shape[0]
    TP = TN = FP1 = FP2 = FN = 0
    for i in range(n):
        for j in range(3):
            if np.amax(y_pred[i][j]) == 0 and np.amax(y_true[i][j]) == 0:
                TN += 1
            elif np.amax(y_pred[i][j]) > 0 and np.amax(y_true[i][j]) == 0:
                FP2 += 1
            elif np.amax(y_pred[i][j]) == 0 and np.amax(y_true[i][j]) > 0:
                FN += 1
            elif np.amax(y_pred[i][j]) > 0 and np.amax(y_true[i][j]) > 0:
                h_pred = (y_pred[i][j] * 255).astype('uint8')
                h_true = (y_true[i][j] * 255).astype('uint8')

                (cnts, _) = cv2.findContours(h_pred.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                rects = [cv2.boundingRect(ctr) for ctr in cnts]
                areas = [r[2] * r[3] for r in rects]
                target = rects[np.argmax(areas)]
                cx_pred = int(target[0] + target[2] / 2)
                cy_pred = int(target[1] + target[3] / 2)

                (cnts, _) = cv2.findContours(h_true.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                rects = [cv2.boundingRect(ctr) for ctr in cnts]
                areas = [r[2] * r[3] for r in rects]
                target = rects[np.argmax(areas)]
                cx_true = int(target[0] + target[2] / 2)
                cy_true = int(target[1] + target[3] / 2)

                dist = math.sqrt((cx_pred - cx_true)**2 + (cy_pred - cy_true)**2)
                if dist > tol:
                    FP1 += 1
                else:
                    TP += 1
    return (TP, TN, FP1, FP2, FN)


def evaluation(outcome_tuple):
    """Compute accuracy, precision, recall from confusion counts."""
    (TP, TN, FP1, FP2, FN) = outcome_tuple
    try:
        accuracy = (TP + TN) / (TP + TN + FP1 + FP2 + FN)
    except ZeroDivisionError:
        accuracy = 0
    try:
        precision = TP / (TP + FP1 + FP2)
    except ZeroDivisionError:
        precision = 0
    try:
        recall = TP / (TP + FN)
    except ZeroDivisionError:
        recall = 0
    return (accuracy, precision, recall)

print("Evaluation helpers defined.")

In [ ]:
# ---- Training Hyperparameters ----

BATCH_SIZE = 10
EPOCHS = 20
LEARNING_RATE = 0.01
TOL = 15             # tolerance in pixels for TP vs FP1
K_LAYERS = 14        # number of conv layers to unfreeze for fine-tuning
TRAIN_VAL_SPLIT = 0.70

print(f"Training config: {EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LEARNING_RATE}, unfreeze last {K_LAYERS} conv layers")

In [ ]:
if not SKIP_TO_INFERENCE:
    # ---- IMPORTANT: Set paths to your weights ----
    # Point this to the old weights folder that was downloaded/unzipped
    # List what's inside WEIGHTS_DIR to find the exact path:
    print("Available weights:")
    for root, dirs, files_list in os.walk(WEIGHTS_DIR):
        level = root.replace(WEIGHTS_DIR, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        for file in files_list:
            print(f"{subindent}{file}")

In [ ]:
if not SKIP_TO_INFERENCE:
    # ========================================================
    # UPDATE THIS PATH to point to the old weights model folder
    # ========================================================
    LOAD_WEIGHTS_PATH = os.path.join(WEIGHTS_DIR, 'old_weights')  # <--- ADJUST if needed
    SAVE_WEIGHTS_PATH = os.path.join(WEIGHTS_DIR, 'fine_tuned')

    print(f"Loading base model from: {LOAD_WEIGHTS_PATH}")
    base_model = load_model(LOAD_WEIGHTS_PATH, custom_objects={'custom_loss': custom_loss})

    # Freeze all layers except last k conv layers
    conv_indices = [idx for idx, layer in enumerate(base_model.layers) if 'conv' in layer.name]
    freeze_to = conv_indices[-K_LAYERS]
    for layer in base_model.layers[:freeze_to]:
        layer.trainable = False
    for layer in base_model.layers[freeze_to:]:
        layer.trainable = True

    model = base_model
    ADADELTA = optimizers.Adadelta(learning_rate=LEARNING_RATE)
    model.compile(loss=custom_loss, optimizer=ADADELTA)

    trainable = np.sum([K.count_params(w) for w in model.trainable_weights])
    non_trainable = np.sum([K.count_params(w) for w in model.non_trainable_weights])
    print(f"Trainable params: {trainable:,}")
    print(f"Non-trainable params: {non_trainable:,}")
    print(f"Frozen up to layer index {freeze_to}, unfreezing last {K_LAYERS} conv layers")

In [ ]:
if not SKIP_TO_INFERENCE:
    # Discover .npy data files
    npy_path = os.path.abspath(DATA_DIR)
    npy_files = glob(os.path.join(npy_path, '*.npy'))
    num_batches = len(npy_files) // 2
    idx = np.arange(num_batches, dtype='int') + 1

    # Compute train/val split indices within each .npy batch
    sample_x = np.load(os.path.join(npy_path, 'x_data_1.npy'))
    npy_batch_size = sample_x.shape[0]
    del sample_x

    val_idx_start = math.floor(TRAIN_VAL_SPLIT * npy_batch_size)
    train = np.arange(0, val_idx_start)
    val = np.arange(val_idx_start, npy_batch_size)

    print(f"Found {num_batches} data batches, each with {npy_batch_size} samples")
    print(f"Train: indices 0-{val_idx_start-1} ({len(train)} samples), Val: indices {val_idx_start}-{npy_batch_size-1} ({len(val)} samples)")

In [ ]:
if not SKIP_TO_INFERENCE:
    # ---- Training Loop ----
    train_metrics = pd.DataFrame(
        columns=["TP", "TN", "FP1", "FP2", "FN", "loss", "acc", "prec", "rec"],
        index=np.arange(1, EPOCHS + 1)
    ).fillna(0)

    val_metrics = pd.DataFrame(
        columns=["TP", "TN", "FP1", "FP2", "FN", "loss", "acc", "prec", "rec"],
        index=np.arange(1, EPOCHS + 1)
    ).fillna(0)

    print('Beginning training...')
    for epoch in range(1, EPOCHS + 1):
        print(f'\n{"="*50} Epoch {epoch}/{EPOCHS} {"="*50}')

        # --- Train ---
        print("TRAINING")
        for file_num in idx:
            X = np.load(os.path.join(DATA_DIR, f'x_data_{file_num}.npy'))
            y = np.load(os.path.join(DATA_DIR, f'y_data_{file_num}.npy'))
            model.fit(X[train], y[train], batch_size=BATCH_SIZE, epochs=1, shuffle=False, verbose=0)
            del X, y
            gc.collect()

        # --- Evaluate ---
        print("EVALUATING")
        for file_num in idx:
            X = np.load(os.path.join(DATA_DIR, f'x_data_{file_num}.npy'))
            y_hat = model.predict(X, batch_size=BATCH_SIZE, verbose=0)
            del X
            gc.collect()

            y = np.load(os.path.join(DATA_DIR, f'y_data_{file_num}.npy'))
            train_metrics.loc[epoch, "loss"] += custom_loss(y[train], y_hat[train]).numpy()
            val_metrics.loc[epoch, "loss"] += custom_loss(y[val], y_hat[val]).numpy()

            y_pred = (y_hat > 0.5).astype("float32")
            train_metrics.loc[epoch, ["TP", "TN", "FP1", "FP2", "FN"]] += outcome(y_pred[train], y[train], TOL)
            val_metrics.loc[epoch, ["TP", "TN", "FP1", "FP2", "FN"]] += outcome(y_pred[val], y[val], TOL)

            del y, y_hat, y_pred
            gc.collect()

        # Compute aggregate metrics
        for metrics_df in [train_metrics, val_metrics]:
            metrics_df.loc[epoch, ["acc", "prec", "rec"]] = evaluation(
                metrics_df.loc[epoch, ["TP", "TN", "FP1", "FP2", "FN"]]
            )
            total = metrics_df.loc[epoch, ["TP", "TN", "FP1", "FP2", "FN"]].sum()
            if total > 0:
                metrics_df.loc[epoch, ["TP", "TN", "FP1", "FP2", "FN"]] /= total
            metrics_df.loc[epoch, ["TP", "TN", "FP1", "FP2", "FN", "acc", "prec", "rec"]] = (
                metrics_df.loc[epoch, ["TP", "TN", "FP1", "FP2", "FN", "acc", "prec", "rec"]] * 100
            ).round(2)

        epoch_summary = pd.concat(
            [train_metrics.loc[epoch, :], val_metrics.loc[epoch, :]],
            axis=1
        )
        epoch_summary.columns = ["train", "val"]
        print(epoch_summary.astype('object'))

        # Save checkpoint
        model.save(f"{SAVE_WEIGHTS_PATH}_epoch{epoch}")
        print(f"  Saved weights to {SAVE_WEIGHTS_PATH}_epoch{epoch}")

    # Save final model
    model.save(SAVE_WEIGHTS_PATH)
    print(f"\nTraining complete! Final weights saved to {SAVE_WEIGHTS_PATH}")

In [ ]:
if not SKIP_TO_INFERENCE:
    # ---- Plot training curves ----
    fig, axes = plt.subplots(3, 3, figsize=(15, 10))
    for i, metric in enumerate(train_metrics.columns):
        ax = axes[i // 3, i % 3]
        x = np.arange(1, EPOCHS + 1)
        ax.plot(x, train_metrics[metric], label='train')
        ax.plot(x, val_metrics[metric], label='val')
        ax.set_title(metric)
        ax.set_xlabel('epoch')
        ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, 'training_curves.png'), dpi=150)
    plt.show()
    print(f"Training curves saved to {os.path.join(WORK_DIR, 'training_curves.png')}")

---
## Step 6: Make Predictions

Runs the trained model on video frames and outputs a CSV with ball coordinates.

**Requires GPU.**

In [ ]:
def run_model_inference(model, unit):
    """Run model inference, handling both Keras 2 models and TFSMLayer."""
    output = model(tf.constant(unit))
    if isinstance(output, dict):
        y_pred = list(output.values())[0].numpy()
    else:
        y_pred = output.numpy() if hasattr(output, 'numpy') else output
    return y_pred


def make_pred_csv(predict_csv_path, video_path, model):
    """Run ball detection on a video and write predictions to CSV."""
    start = time.time()
    f = open(predict_csv_path, 'w')
    f.write('Frame,Visibility,X,Y\n')

    cap = cv2.VideoCapture(video_path)
    cap.set(1, 0)
    success, image1 = cap.read()
    success, image2 = cap.read()
    success, image3 = cap.read()
    ratio = image1.shape[0] / HEIGHT
    num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    count = 0

    while success:
        unit = []
        for img in [image1, image2, image3]:
            x = img[..., ::-1]  # BGR -> RGB
            x = array_to_img(x)
            x = x.resize(size=(WIDTH, HEIGHT))
            x = np.moveaxis(img_to_array(x), -1, 0)
            unit.extend([x[0], x[1], x[2]])

        unit = np.asarray(unit).reshape((1, 9, HEIGHT, WIDTH)).astype('float32') / 255
        y_pred = run_model_inference(model, unit)
        y_pred = (y_pred > 0.5).astype('float32')
        h_pred = (y_pred[0] * 255).astype('uint8')

        for i, image in enumerate([image1, image2, image3]):
            if np.amax(h_pred[i]) <= 0:
                f.write(f'{count},0,-1,-1\n')
            else:
                (cnts, _) = cv2.findContours(h_pred[i].copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                rects = [cv2.boundingRect(ctr) for ctr in cnts]
                areas = [r[2] * r[3] for r in rects]
                target = rects[np.argmax(areas)]
                cx = int(ratio * (target[0] + target[2] / 2))
                cy = int(ratio * (target[1] + target[3] / 2))
                f.write(f'{count},1,{cx},{cy}\n')
            count += 1

        if count % 300 == 0:
            elapsed = time.time() - start
            pct = count / num_frames * 100
            print(f'  {count}/{num_frames} frames ({pct:.1f}%) — {elapsed:.1f}s elapsed')

        success, image1 = cap.read()
        success, image2 = cap.read()
        success, image3 = cap.read()

    f.close()
    cap.release()
    elapsed = time.time() - start
    print(f'  Done! Predicted {count}/{num_frames} frames in {elapsed:.1f}s')
    return count

print("Prediction function defined.")

In [ ]:
# ---- Load model for prediction ----
# Uses TFSMLayer to load legacy SavedModel format (compatible with Keras 3 / TF 2.16+)
import tensorflow as tf

if SKIP_TO_INFERENCE or not os.path.exists(SAVE_WEIGHTS_PATH):
    PRED_WEIGHTS_PATH = NEW_WEIGHTS_PATH
    print(f"Loading pre-trained new weights from: {PRED_WEIGHTS_PATH}")
    print("Available files in new weights dir:")
    !ls -la {PRED_WEIGHTS_PATH}
else:
    PRED_WEIGHTS_PATH = SAVE_WEIGHTS_PATH
    print(f"Using fine-tuned weights from training: {PRED_WEIGHTS_PATH}")

pred_model = tf.keras.layers.TFSMLayer(PRED_WEIGHTS_PATH, call_endpoint='serving_default')
print("Model loaded for prediction.")

In [ ]:
# ---- Run Predictions ----
PRED_CSV_PATH = os.path.join(PREDICTIONS_DIR, f'{VIDEO_FILENAME[:-4]}_predictions.csv')

print(f"Running predictions on {VIDEO_FILENAME}...")
n_predicted = make_pred_csv(PRED_CSV_PATH, VIDEO_PATH, pred_model)

print(f"\nPrediction CSV saved to: {PRED_CSV_PATH}")
print(f"\nPrediction preview:")
print(pd.read_csv(PRED_CSV_PATH).head(20))

---
## Step 7: Model Performance Evaluation

Compares predictions against ground truth labels to compute accuracy, precision, and recall.

**Requires label CSV (skip if you don't have ground truth labels).**

In [ ]:
def calc_frame_metrics(y_pred, x_pred, y_true, x_true, scaled_tol):
    """Compute confusion matrix for a single frame."""
    TP = TN = FP1 = FP2 = FN = 0
    if x_pred < 0 and y_pred < 0 and x_true < 0 and y_true < 0:
        TN = 1
    elif x_pred > 0 and y_pred > 0 and x_true < 0 and y_true < 0:
        FP2 = 1
    elif x_pred < 0 and y_pred < 0 and x_true > 0 and y_true > 0:
        FN = 1
    elif x_true > 0 and y_true > 0 and x_pred > 0 and y_pred > 0:
        dist = math.sqrt((x_pred - x_true)**2 + (y_pred - y_true)**2)
        if dist > scaled_tol:
            FP1 = 1
        else:
            TP = 1
    return np.array((TP, TN, FP1, FP2, FN))


def calc_all_metrics(df, scaled_tol):
    """Compute metrics across all frames."""
    totals = np.array([0, 0, 0, 0, 0])
    for y_pred, x_pred, y_true, x_true in zip(
        df["Y_Predicted"], df["X_Predicted"], df["Y_True"], df["X_True"]
    ):
        totals += calc_frame_metrics(y_pred, x_pred, y_true, x_true, scaled_tol)

    TP, TN, FP1, FP2, FN = totals
    n = np.sum(totals)

    try: acc = (TP + TN) / n
    except: acc = 0
    try: prec = TP / (TP + FP1 + FP2)
    except: prec = 0
    try: rec = TP / (TP + FN)
    except: rec = 0

    return np.round(100 * np.array([
        TP/n, TN/n, FP1/n, FP2/n, FN/n, acc, prec, rec
    ]), 2)

print("Performance evaluation functions defined.")

In [ ]:
HAS_LABELS = not SKIP_TO_INFERENCE and os.path.exists(FIXED_LABEL_PATH)

if HAS_LABELS:
    TOL_FRACTION = 0.0075  # tolerance as fraction of video diagonal

    # Get video dimensions
    cap = cv2.VideoCapture(VIDEO_PATH)
    vid_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    vid_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    scaled_tol = TOL_FRACTION * math.sqrt(vid_height**2 + vid_width**2)

    # Load labels
    labels = pd.read_csv(FIXED_LABEL_PATH)
    label_coords = labels[['X', 'Y']].copy()
    label_coords.columns = ['X_True', 'Y_True']

    # Load predictions
    preds = pd.read_csv(PRED_CSV_PATH)
    pred_coords = preds[['X', 'Y']].copy()
    pred_coords.columns = ['X_Predicted', 'Y_Predicted']

    # Merge (align by frame index)
    n_common = min(len(label_coords), len(pred_coords))
    df_eval = pd.concat([label_coords.iloc[:n_common], pred_coords.iloc[:n_common]], axis=1)

    metrics = calc_all_metrics(df_eval, scaled_tol)
    metric_names = ['TP%', 'TN%', 'FP1%', 'FP2%', 'FN%', 'Accuracy', 'Precision', 'Recall']

    print(f"\nPerformance Metrics (tolerance={scaled_tol:.2f}px)")
    print("=" * 40)
    for name, val in zip(metric_names, metrics):
        print(f"  {name:>12}: {val:.2f}%")

    perf_df = pd.DataFrame([metrics], columns=metric_names)
    perf_df.to_csv(os.path.join(WORK_DIR, 'performance_metrics.csv'), index=False)
    print(f"\nSaved to {os.path.join(WORK_DIR, 'performance_metrics.csv')}")
else:
    print("No ground truth labels available. Skipping performance evaluation.")
    print("You can still generate trajectory videos in the next step.")

---
## Step 8: Trajectory Overlay on Video

Draws the predicted ball positions on the video as a trailing trajectory.

In [ ]:
def get_trajectory(predict_csv_path, video_path, output_video_path, trail_length=8):
    """Overlay ball trajectory on video."""
    # Read predictions
    preds = pd.read_csv(predict_csv_path)
    x_coords = preds['X'].values
    y_coords = preds['Y'].values
    num_preds = len(x_coords)

    # Trajectory queue
    q = queue.deque([None] * trail_length)

    # Video setup
    video = cv2.VideoCapture(video_path)
    fps = int(video.get(cv2.CAP_PROP_FPS))
    num_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    out_w = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    out_h = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    output_video = cv2.VideoWriter(output_video_path, fourcc, fps, (out_w, out_h))

    currentFrame = 0
    video.set(1, 0)

    # Write first two frames as-is
    for _ in range(2):
        ret, img = video.read()
        if ret:
            output_video.write(img)
            currentFrame += 1

    while currentFrame < num_preds and currentFrame < num_frames:
        video.set(1, currentFrame)
        ret, img = video.read()
        if not ret:
            break

        pil_img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        if x_coords[currentFrame] > 0 and y_coords[currentFrame] > 0:
            q.appendleft([int(x_coords[currentFrame]), int(y_coords[currentFrame])])
        else:
            q.appendleft(None)
        q.pop()

        draw = ImageDraw.Draw(pil_img)
        for i in range(trail_length):
            if q[i] is not None:
                dx, dy = q[i]
                radius = max(2, trail_length - i)
                bbox = (dx - radius, dy - radius, dx + radius, dy + radius)
                draw.ellipse(bbox, outline='yellow', fill='yellow')
        del draw

        output_video.write(cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR))
        currentFrame += 1

        if currentFrame % 500 == 0:
            print(f'  Processed {currentFrame}/{min(num_preds, num_frames)} frames')

    video.release()
    output_video.release()
    print(f'  Done! Trajectory video: {output_video_path} ({currentFrame} frames)')

print("Trajectory function defined.")

In [ ]:
# ---- Generate Trajectory Video ----
OUTPUT_VIDEO_PATH = os.path.join(TRAJECTORY_DIR, f'{VIDEO_FILENAME[:-4]}_trajectory.mp4')

print(f"Generating trajectory video for {VIDEO_FILENAME}...")
get_trajectory(PRED_CSV_PATH, VIDEO_PATH, OUTPUT_VIDEO_PATH)
print(f"\nTrajectory video saved to: {OUTPUT_VIDEO_PATH}")

In [ ]:
# ---- Preview trajectory video (first few seconds) ----
from IPython.display import HTML
from base64 import b64encode

# Read a small portion for preview
cap = cv2.VideoCapture(OUTPUT_VIDEO_PATH)
fps = int(cap.get(cv2.CAP_PROP_FPS))
preview_frames = fps * 10  # 10 seconds preview
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

preview_path = os.path.join(TRAJECTORY_DIR, 'preview.mp4')
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(preview_path, fourcc, fps, (w, h))

for i in range(preview_frames):
    ret, frame = cap.read()
    if not ret:
        break
    out.write(frame)

cap.release()
out.release()

# Display in notebook
mp4 = open(preview_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=640 controls><source src="{data_url}" type="video/mp4"></video>')

---
## 9. Download Results

In [ ]:
# Download the prediction CSV
from google.colab import files
files.download(PRED_CSV_PATH)

In [ ]:
# Download the trajectory video
files.download(OUTPUT_VIDEO_PATH)

In [ ]:
# Save everything to Google Drive for persistence
DRIVE_SAVE_DIR = '/content/drive/MyDrive/TrackNet_Results'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Copy results
shutil.copy(PRED_CSV_PATH, DRIVE_SAVE_DIR)
shutil.copy(OUTPUT_VIDEO_PATH, DRIVE_SAVE_DIR)
if not SKIP_TO_INFERENCE and os.path.exists(SAVE_WEIGHTS_PATH):
    shutil.copytree(SAVE_WEIGHTS_PATH, os.path.join(DRIVE_SAVE_DIR, 'fine_tuned_weights'), dirs_exist_ok=True)

print(f"All results saved to Google Drive at: {DRIVE_SAVE_DIR}")
!ls -la {DRIVE_SAVE_DIR}

---
## Summary

| Step | What it does | Status |
|------|-------------|--------|
| 0 | Environment setup | Done |
| 1 | Video → Frames | Done |
| 2-3 | Labelling + Fix labels | Done (or skipped) |
| 4 | Frames → .npy training data | Done (or skipped) |
| 5 | Transfer learning (training) | Done (or skipped) |
| 6 | Video → Prediction CSV | Done |
| 7 | Performance evaluation | Done (or skipped) |
| 8 | Trajectory overlay video | Done |

### Key Metrics:
- **TP**: Predicts ball at correct location
- **TN**: Correctly predicts no ball
- **FP1**: Predicts ball at wrong location
- **FP2**: Predicts ball when there is none
- **FN**: Misses the ball